# H10 · Shocks: did stronger institutions cushion the COVID crash and the 2022 energy shock?

> **Hypothesis H10:** Listed companies in countries with stronger institutions fell less in the COVID crash of early 2020.

| | |
|---|---|
| **Why we ask** | Averages hide what matters in a crisis. Strong institutions (rule of law, effective government) should let a country absorb a sudden shock with less panic and better policy response. A crash is a natural experiment: every country is hit at once, so differences between countries show who was more exposed. |
| **Prediction** | **Positive** association between the institutions index at the end of 2019 and the stock return from end of January to end of March 2020 (less negative = better). |
| **Data** | Monthly adjusted close prices of the 2,590 Yahoo companies. Crash return = March 2020 close over January 2020 close. Institutions index = average of six governance indicators (2019 values), in SDs across countries. Size = rank of market capitalisation within country. |
| **Primary test (fixed before looking at the result)** | `crash return (pp) ~ institutions index + sector + size rank`, errors clustered by country, wild-bootstrap p-value. |
| **Verdict rule** | *Supported* if the effect has the predicted sign, the wild-bootstrap p < 0.05, and the sign holds in at least 80% of the robustness samples. *Partly supported* if it has the predicted sign and p < 0.10 (or p < 0.05 but robustness is below 80%). *Contradicted* if significantly the opposite way. Otherwise *not supported*. The shock is one episode, so the result is about early 2020 and nothing more general. |

**Honest framing.** Several hypotheses in this series (H1 to H3) were suggested by an exploratory scan of the same data, so their p-values are optimistic and a pass is a lead, not a confirmation. Where possible the result is **replicated on ESEF**, an independent source (which still shares some companies with Yahoo). All hypotheses are corrected for multiple testing together in `1.0_hypotheses.ipynb`. Everything is **association, not causation**.

In [1]:
import sys, pathlib, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore")
ROOT = pathlib.Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src" / "analysis"))
import prep, utils
from utils import CONTROLS, fit_cluster, wild_cluster, spearman, boot_ci, partial_rho, bh, scan

con = prep.connect()
q = lambda sql: con.sql(sql).df()
pd.set_option("display.width", 220, "display.max_columns", 40, "display.max_rows", 120, "display.float_format", "{:,.3f}".format)
px.defaults.template = "plotly_white"

country = prep.load_country(con)
dim, profile, profile_all, panel = country["dim"], country["profile"], country["profile_all"], country["panel"]
feat_cols = country["feat_cols"]
fy = prep.load_firms(con, country)               # Yahoo company-years (primary sample)
names = dim.name
MIN_FIRMS = prep.MIN_FIRMS
print(f"{len(dim)} countries | {len(feat_cols)} country features | {len(fy):,} Yahoo company-years of {fy.company_id.nunique():,} companies")

34 countries | 35 country features | 10,220 Yahoo company-years of 2,590 companies


## 1 · The crash in the data
Median price path of listed companies in countries with high, middle and low institutions (terciles of the 2019 institutions index), January 2020 = 100.

In [2]:
P = pd.read_parquet(ROOT / "data" / "flat" / "prices_monthly.parquet")
P = P.pivot(index="month", columns="ticker", values="adj_close")
comp = q("SELECT company_id, source_key AS ticker, iso3, sector, market_cap FROM dim_company WHERE source_id = 'YF'")
comp["sector"] = comp.sector.fillna("Unknown"); comp["size_rank"] = comp.groupby("iso3").market_cap.rank(pct=True)
lag = prep.country_year_lagged(country)
l20 = lag[lag.year == 2020].set_index("iso3")
comp = comp.join(l20[["inst_z", "dev_z", "gdp_growth_z", "gov_debt_gdp_z", "private_credit_gdp_z", "trade_gdp_z"]], on="iso3")
comp = comp[comp.ticker.isin(P.columns)].set_index("ticker")
m = lambda s: pd.Timestamp(s)
def window(a, b):
    return (P.loc[m(b)] / P.loc[m(a)] - 1) * 100
comp["crash"] = window("2020-01-01", "2020-03-01")             # end Jan -> end Mar 2020
comp["crash_feb"] = window("2020-02-01", "2020-03-01")         # end Feb -> end Mar (cleaner, shorter)
comp["crash_apr"] = window("2020-01-01", "2020-04-01")         # to end April
comp["recovery"] = window("2020-03-01", "2020-12-01")          # end Mar -> end Dec 2020
for c in ["crash", "crash_feb", "crash_apr", "recovery"]:
    lo, hi = comp[c].quantile(prep.WINSOR); comp[c] = comp[c].clip(lo, hi)
comp = comp.reset_index()
comp["tercile"] = pd.qcut(comp.inst_z, 3, labels=["low institutions", "middle", "high institutions"])
base = P.loc[m("2020-01-01")]
path = (P.loc["2019-12-01":"2021-03-01"] / base * 100).stack().rename("idx").reset_index().merge(comp[["ticker", "tercile"]], on="ticker")
path = path.groupby(["month", "tercile"], observed=True).idx.median().unstack()
fig = px.line(path, title="Median price path, January 2020 = 100, by institutions tercile", labels={"value": "index", "month": ""}); fig.add_vline(x=pd.Timestamp("2020-03-01"), line_dash="dot")
fig.update_layout(height=420); fig.show()
print(f"{comp.crash.notna().sum():,} companies with prices in Jan and Mar 2020, in {comp.dropna(subset=['crash']).iso3.nunique()} countries. Median crash {comp.crash.median():.1f}%, recovery to December {comp.recovery.median():.1f}%.")

2,232 companies with prices in Jan and Mar 2020, in 34 countries. Median crash -21.8%, recovery to December 32.2%.


## 2 · Primary test

In [3]:
cd = comp.dropna(subset=["crash", "inst_z", "size_rank"]).copy(); cd["fiscal_year"] = 2020
primary = wild_cluster("inst_z", cd, outcome="crash", controls="C(sector) + size_rank", B=1999)
print(f"{primary['n']:,} companies, {primary['clusters']} countries. Typical crash: {cd.crash.median():.1f}%.")
print(f"+1 SD institutions → {primary['coef']:+.2f} pp crash return (95% CI {primary['ci_low']:+.2f} to {primary['ci_high']:+.2f}); a positive number means a smaller fall.")
print(f"p (cluster-robust) = {primary['p_cluster']:.4f} | p (wild bootstrap) = {primary['p_wild']:.4f}")

2,227 companies, 34 countries. Typical crash: -21.8%.
+1 SD institutions → +0.02 pp crash return (95% CI -2.56 to +2.60); a positive number means a smaller fall.
p (cluster-robust) = 0.9864 | p (wild bootstrap) = 0.9880


### What protected firms? Country-level scan
Median crash return per country against every country feature (Spearman, Benjamini–Hochberg corrected). Countries with fewer than 10 companies are left out.

In [4]:
cc = comp.dropna(subset=["crash"]).groupby("iso3").agg(crash=("crash", "median"), n=("crash", "size")).query("n >= @MIN_FIRMS")
sc = scan(profile.loc[cc.index], cc.crash, feat_cols, min_n=15)
fig = px.bar(sc.reset_index().head(12), x="rho", y="feature", orientation="h", color=sc.head(12).q.values < 0.10, title="Strongest country features linked to the March 2020 crash (Spearman ρ, country medians)",
             labels={"color": "q < 0.10"}); fig.update_layout(height=430, yaxis_autorange="reversed"); fig.show()
print(f"{len(sc)} features scanned; {int((sc.q < 0.10).sum())} with q < 0.10. A positive ρ = smaller fall where the feature is high.")
sc.head(12)

35 features scanned; 3 with q < 0.10. A positive ρ = smaller fall where the feature is high.


,rho,p,n,q
feature,,,,
youth_unemployment,-0.498,0.003,33,0.069
manufacturing_share,0.476,0.005,33,0.069
unemployment,-0.469,0.006,33,0.069
rd_gdp,0.346,0.061,30,0.380
tax_revenue_gdp,-0.341,0.065,30,0.380
hightech_exports,0.329,0.061,33,0.380
labour_participation,0.228,0.201,33,0.924
gov_debt_gdp,-0.214,0.231,33,0.924
services_share,-0.186,0.300,33,0.924


### Recovery and the 2022 energy shock
Two more episodes, reported as secondary results (corrected together). *Recovery:* end-March to end-December 2020. *Energy shock:* real return in calendar 2022 against how energy-intensive the economy was (energy use per unit of GDP, 2019, from World Bank), expected negative.

In [5]:
E = pd.read_parquet(ROOT / "data" / "flat" / "extra_country_year.parquet")
ei = E[(E.indicator == "energy_intensity") & E.year.between(2017, 2019)].groupby("iso3").value.mean()
ei_z = ((np.log(ei) - np.log(ei).mean()) / np.log(ei).std()).rename("energy_z")
dec = P.loc[[pd.Timestamp("2021-12-01"), pd.Timestamp("2022-12-01")]]
r22 = ((dec.iloc[1] / dec.iloc[0] - 1) * 100).rename("ret22").reset_index()
r22 = r22.merge(comp[["ticker", "iso3", "sector", "size_rank"]], on="ticker").merge(panel[panel.year == 2022][["iso3", "inflation"]], on="iso3")
r22["real22"] = ((1 + r22.ret22 / 100) / (1 + r22.inflation / 100) - 1) * 100
lo, hi = r22.real22.quantile(prep.WINSOR); r22["real22"] = r22.real22.clip(lo, hi)
r22 = r22.join(ei_z, on="iso3").dropna(subset=["real22", "energy_z", "size_rank"]); r22["fiscal_year"] = 2022
rows = []
for lab, out, ft, data, ctrl in [("COVID crash, Jan→Mar 2020  [primary]", "crash", "inst_z", cd, "C(sector) + size_rank"), ("recovery, Mar→Dec 2020", "recovery", "inst_z", cd.dropna(subset=["recovery"]), "C(sector) + size_rank"),
                                   ("2022 real return vs energy intensity", "real22", "energy_z", r22, "C(sector) + size_rank")]:
    r = wild_cluster(ft, data, outcome=out, controls=ctrl, B=999)
    rows.append({"episode": lab, "country feature": ft, "coef per +1 SD (pp)": r["coef"], "ci_low": r["ci_low"], "ci_high": r["ci_high"], "p (wild)": r["p_wild"], "n": r["n"], "countries": r["clusters"]})
secondary = pd.DataFrame(rows).set_index("episode"); secondary["q (BH)"] = bh(secondary["p (wild)"])
secondary

,country feature,coef per +1 SD (pp),ci_low,ci_high,p (wild),n,countries,q (BH)
episode,,,,,,,,
"COVID crash, Jan→Mar 2020 [primary]",inst_z,0.023,-2.557,2.602,0.988,2227,34,0.988
"recovery, Mar→Dec 2020",inst_z,-1.199,-7.069,4.672,0.710,2227,34,0.988
2022 real return vs energy intensity,energy_z,0.177,-2.168,2.522,0.870,2372,34,0.988


## 3 · Robustness
The primary test in the usual samples (the time-window sample does not apply to a single episode) and with other definitions of the crash window.

In [6]:
SM = {"all firms": lambda d: d, "excluding financials & real estate": lambda d: d[~d.sector.isin(["Financial Services", "Real Estate"])],
      "excluding small-sample countries": lambda d: d[~d.iso3.isin(dim.index[dim.small_firm_sample])], "excluding USA, China, Japan": lambda d: d[~d.iso3.isin(["USA", "CHN", "JPN"])],
      "EU countries only": lambda d: d[d.iso3.isin(dim.index[dim.is_eu])], "non-EU countries only": lambda d: d[~d.iso3.isin(dim.index[dim.is_eu])]}
rows = []
for nm, f in SM.items():
    r = wild_cluster("inst_z", f(cd), outcome="crash", controls="C(sector) + size_rank", B=499)
    rows.append({"variant": nm, "coef": r["coef"], "ci_low": r["ci_low"], "ci_high": r["ci_high"], "p (wild)": r["p_wild"], "countries": r["clusters"], "n": r["n"]})
for nm, out in [("window Feb→Mar 2020", "crash_feb"), ("window Jan→Apr 2020", "crash_apr")]:
    d = comp.dropna(subset=[out, "inst_z", "size_rank"]).assign(fiscal_year=2020)
    r = wild_cluster("inst_z", d, outcome=out, controls="C(sector) + size_rank", B=499)
    rows.append({"variant": nm, "coef": r["coef"], "ci_low": r["ci_low"], "ci_high": r["ci_high"], "p (wild)": r["p_wild"], "countries": r["clusters"], "n": r["n"]})
rob = pd.DataFrame(rows).set_index("variant"); rob_share = float((rob.coef > 0).mean())
fig = px.scatter(rob.reset_index(), x="coef", y="variant", error_x=rob.ci_high - rob.coef, error_x_minus=rob.coef - rob.ci_low, title="Institutions → crash return (pp per +1 SD), variants")
fig.add_vline(x=0, line_color="grey"); fig.update_layout(height=420, yaxis_autorange="reversed"); fig.show()
print(f"Predicted sign (positive) in {rob_share:.0%} of variants.")
rob

Predicted sign (positive) in 75% of variants.


,coef,ci_low,ci_high,p (wild),countries,n
variant,,,,,,
all firms,0.023,-2.557,2.602,0.986,34,2227
excluding financials & real estate,-0.258,-2.753,2.237,0.886,34,1751
excluding small-sample countries,0.115,-2.471,2.702,0.942,29,2135
"excluding USA, China, Japan",0.978,-0.793,2.750,0.358,31,1949
EU countries only,1.217,-1.457,3.891,0.472,21,1108
non-EU countries only,-0.007,-3.442,3.428,1.000,13,1119
window Feb→Mar 2020,0.301,-1.895,2.498,0.766,34,2232
window Jan→Apr 2020,0.922,-1.429,3.272,0.478,34,2227


## 4 · Replication on accounts (ESEF)
A different outcome and source: the **change in ROA from fiscal 2019 to 2020** of EU companies (ESEF), against the institutions index of 2019. Expected positive (a smaller fall in profitability where institutions are strong).

In [7]:
fe = prep.load_firms(con, country, source="ESEF")
w = fe[fe.fiscal_year.isin([2019, 2020])].pivot_table(index="company_id", columns="fiscal_year", values="roa_pp")
w = w.dropna(subset=[2019, 2020]); w["d_roa"] = w[2020] - w[2019]
meta = fe[fe.fiscal_year == 2020].drop_duplicates("company_id").set_index("company_id")[["iso3", "size_rank", "inst_z"]]
dw = w.join(meta, how="inner").dropna(subset=["inst_z", "size_rank"]).reset_index(); dw["fiscal_year"] = 2020
rep = wild_cluster("inst_z", dw, outcome="d_roa", controls="size_rank", B=1999)
rep_same = bool(np.sign(rep["coef"]) == 1)
print(f"ESEF ({rep['n']:,} companies with ROA in 2019 and 2020, {rep['clusters']} countries): +1 SD institutions → {rep['coef']:+.2f} pp change in ROA (95% CI {rep['ci_low']:+.2f} to {rep['ci_high']:+.2f}), wild p = {rep['p_wild']:.3f}")
print(f"Typical change in ROA 2019→2020: {dw.d_roa.median():+.2f} pp.")

ESEF (70 companies with ROA in 2019 and 2020, 10 countries): +1 SD institutions → +0.96 pp change in ROA (95% CI -0.14 to +2.06), wild p = 0.192
Typical change in ROA 2019→2020: -0.69 pp.


## 5 · Verdict

In [8]:
v = utils.verdict(primary["coef"], primary["p_wild"], 1, rob_share, primary["clusters"])
top = sc.iloc[0]
lines = [
 f"Primary: +1 SD institutions → {primary['coef']:+.2f} pp return in the Jan–Mar 2020 crash (typical {cd.crash.median():.1f}%), wild p = {primary['p_wild']:.3f}, {primary['clusters']} countries, {primary['n']:,} companies.",
 f"Robustness: predicted sign in {rob_share:.0%} of variants. ESEF accounts (change in ROA 2019→2020): {rep['coef']:+.2f} pp (wild p = {rep['p_wild']:.3f}), " + ("same direction." if rep_same else "opposite direction."),
 f"Country scan: strongest feature for the crash is {top.name} (ρ = {top.rho:+.2f}, q = {top.q:.2f}); {int((sc.q < 0.10).sum())} of {len(sc)} features have q < 0.10.",
 f"Recovery (Mar→Dec 2020): {secondary.iloc[1]['coef per +1 SD (pp)']:+.1f} pp (wild p = {secondary.iloc[1]['p (wild)']:.3f}); 2022 real return vs energy intensity: {secondary.iloc[2]['coef per +1 SD (pp)']:+.1f} pp (wild p = {secondary.iloc[2]['p (wild)']:.3f}).",
 f"VERDICT H10: {v.upper()}",
]
print("\n".join(lines))
utils.save_result("H10", id="H10", title="Shocks: COVID crash and institutions", statement="Companies in countries with stronger institutions fell less in the early-2020 crash.",
                  expected_sign=1, effect=primary["coef"], ci_low=primary["ci_low"], ci_high=primary["ci_high"], effect_unit="pp crash return per +1 SD institutions (Jan–Mar 2020)",
                  p_primary=primary["p_wild"], p_method="wild cluster bootstrap", n=primary["n"], clusters=primary["clusters"], robust_share=rob_share,
                  replication_coef=rep["coef"], replication_p=rep["p_wild"], replication_same_sign=rep_same, verdict=v,
                  notebook="1.10_shocks_covid_and_energy.ipynb", notes=lines)

Primary: +1 SD institutions → +0.02 pp return in the Jan–Mar 2020 crash (typical -21.8%), wild p = 0.988, 34 countries, 2,227 companies.
Robustness: predicted sign in 75% of variants. ESEF accounts (change in ROA 2019→2020): +0.96 pp (wild p = 0.192), same direction.
Country scan: strongest feature for the crash is youth_unemployment (ρ = -0.50, q = 0.07); 3 of 35 features have q < 0.10.
Recovery (Mar→Dec 2020): -1.2 pp (wild p = 0.710); 2022 real return vs energy intensity: +0.2 pp (wild p = 0.870).
VERDICT H10: NOT SUPPORTED


### Limits
- **One episode.** The crash lasted weeks and is dominated by how severely each market was hit and how fast governments intervened; institutions are only one of many correlated factors.
- **Sector mix and timing.** Sector controls help, but countries differ in the timing of the first outbreak, so the result is not a clean test of institutions.
- **Local currency, survivor bias.** Prices are in local currency and delisted firms are missing.
- **ESEF replication is a different outcome** (accounting ROA, a year-on-year change, EU only); it checks the direction, not the size.
- Association, not causation.

In [9]:
con.close()